## Настройка среды

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, regexp_tokenize

In [ ]:
# ! mkdir -p ../data/raw
# ! curl -L -o ../data/raw/myanimelist-comment-dataset-v2.zip \
#     https://www.kaggle.com/api/v1/datasets/download/natlee/myanimelist-comment-dataset-v2
# ! unzip -d ../data/raw -o ../data/raw/myanimelist-comment-dataset-v2.zip

In [ ]:
RANDOM_STATE = 13112026

PROJECT_ROOT = Path.cwd().parent
DATASET_PATH = PROJECT_ROOT / 'data' / 'raw'

In [ ]:
nltk.download('stopwords')

---

## Загрузка и первичная очистка

In [ ]:
df_reviews = pd.read_csv(DATASET_PATH / 'reviews.csv', header=0)
df_al = pd.read_csv(DATASET_PATH / 'anime_list.csv', header=0)

In [ ]:
df_reviews.head(3)

In [ ]:
df_al.head(3)

In [ ]:
df = pd.merge(df_reviews, df_al, on='workId', how='left')

In [ ]:
del df_reviews, df_al

In [ ]:
df.head(3)

In [ ]:
df.columns

In [ ]:
drop_cols = [
    'id_x', 'workId', 'url_x', 'postTime', 'episodesSeen',
    'author', 'reviewId', 'nice', 'loveIt', 'funny',
    'confusing', 'informative', 'wellWritten', 'creative', 'id_y', 'url_y', 
    'jpName', 'workType', 'episodes', 'status', 'aired', 'premiered', 'producer',
    'broadcast', 'licensors', 'studios', 'genres', 'themes', 'demographic',
    'source', 'duration', 'rating', 'score', 'allRank', 'popularityRank',
    'members', 'favorites', 'scoredByUser', 'lastUpdate'
]

df = df.drop(columns=drop_cols)
df.head(3)

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
no_recommendation_status_mask = df['recommendationStatus'].notna()

**Подготовка двух потенциальных таргетов датасета**

In [ ]:
df['overallRating'].value_counts()

In [ ]:
# df = df[df['overallRating'] != 0]
df = df.query('overallRating != 0')
np.sort(df['overallRating'].unique())

In [ ]:
df['recommendationStatus'].value_counts()

In [ ]:
rec_bin_col = df['recommendationStatus'].notna()
df['recommendationStatus'] = df['recommendationStatus'].astype(bool)
df['recommendationStatus'] = rec_bin_col
df['recommendationStatus'].value_counts()

---

---

## 1.

Загрузите ваш индивидуальный датасет. Выделите целевую текстовую колонку. Разделите выборку на обучающую (Train) и тестовую (Test) в соотношении 80/20 с применением стратификации по целевому классу.

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['recommendationStatus'], random_state=RANDOM_STATE)
df_train.shape, df_test.shape

---

## 2.

Исследуйте структуру данных строго на обучающей выборке: подсчитайте общее количество документов, среднюю длину текстов в словах и символах. Постройте графики распределения длин текстов и частоты топ-30 стоп-слов.

In [ ]:
def investigate_df(df, text_col: str = 'review'):
    docs: pd.Series = df[text_col]
    
    # Общее количество документов
    num_docs = docs.shape[0]
    print(f'Number of documents: {num_docs}')
    
    # Средняя длина текстов в словах и в символах
    # text_lengths_in_words = docs.map(lambda s: len(s.split()))
    text_lengths_in_words = docs.map(lambda s: len(regexp_tokenize(s, r'\w+')))
    text_lengths_in_signs = docs.map(lambda s: len(s))
    
    mean_text_lengths_in_words = text_lengths_in_words.mean()
    mean_text_lengths_in_signs = text_lengths_in_signs.mean()
    print(f'Mean text length (in words): {mean_text_lengths_in_words}')
    print(f'Mean text length (in signs): {mean_text_lengths_in_signs}')

    # Частота встречаемости топ-30 английских стоп-слов
    stopwords_counter = Counter()
    stops = set(stopwords.words('english'))
    
    for s in docs:
        for token in regexp_tokenize(s, r'\w+'):
            if token in stops:
                stopwords_counter[token] += 1

    # TODO: Построить график
    # Использовать Counter.most_common(n=30)

---

## 3.

Проанализируйте текст и выявите не менее 3 типов специфического шума, характерного для вашего домена (например: HTML-теги, URL-адреса, Twitter-ники вроде @username, технические логи, шестнадцатеричные символы, смайлики или спецсимволы). Шума может и не быть в датасете, докажите это.

---

## 4.

Напишите комплексный конвейер очистки текста, используя модуль re (Регулярные выражения). Запрещено просто удалять всё, кроме букв. Каждый тип шума должен обрабатываться своим правилом (например, URL-адреса должны заменяться на специальный токен `[URL]`, а числа — на токен `[NUM]`, либо аккуратно вырезаться, если они не несут смысла для вашей задачи). На выходе конвейера добавьте к каждому документу технические токены начала и конца последовательности: `<s>` и `</s>`.

---

## 5.

Сравнительная нормализация: Проведите нормализацию текстов двумя способами: с помощью стемминга и с помощью лемматизации. Сравните размер полученных словарей (количества уникальных токенов) до и после каждого вида нормализации. Сделайте выводы о применимости каждого метода для вашего типа текстов.

---

## 6.

Поиск опечаток через строковые метрики: Самостоятельно (без использования готовых библиотек вычисления расстояния) реализуйте алгоритм расчета расстояния Левенштейна. Оптимизируйте алгоритм по памяти. С его помощью найдите в корпусе примеры слов с расстоянием редактирования равным 1 или 2 (опечатки, разные формы одного слова) и выведите топ-10 пар, если такие опечатки существуют.

---

## 7.

Построение языковой модели: На основе очищенного и токенизированного текста обучающей выборки постройте статистическую \(N\)-граммную модель (\(N=2\) или \(N=3\)).Сформируйте явный словарь модели и предусмотрите обработку неизвестных слов из тестовой выборки с помощью специального токена `[UNK]`.

---

## 8.

Сглаживание вероятностей: Реализуйте один/два из классических методов сглаживания для корректной обработки \(N\)-грамм, отсутствующих в обучающей выборке. Напишите функцию расчета метрики Perplexity (перплексия) на основе логарифма правдоподобия (log-likelihood), чтобы избежать численного переполнения. Рассчитайте и сравните перплексию построенной модели на тестовой выборке (Test) в двух режимах: без сглаживания и со сглаживанием. Сделайте математически обоснованный вывод в отчете.

---

## 9.

Используя построенную модель на базе Марковских цепей, сгенерируйте 5 случайных текстов. Генерация должна начинаться с токена `<s>` и завершаться при достижении токена `</s>` или максимальной заданной длины. Оцените их грамматическую связность и смысловую адекватность в отчете.

---